# 🎮 GPU Cloud Server — Cloud Gaming + Treo Bot trên Kaggle / Google Colab

<a href="https://colab.research.google.com/github/tiendatmeow-source/cloud-gaming-gpu/blob/main/cloud_gpu_server.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Ftiendatmeow-source%2Fcloud-gaming-gpu%2Fblob%2Fmain%2Fcloud_gpu_server.ipynb"><img src="https://img.shields.io/badge/Open%20in%20Kaggle-20BEFF?logo=kaggle&logoColor=white" alt="Open In Kaggle"/></a>
<a href="https://github.com/tiendatmeow-source/cloud-gaming-gpu"><img src="https://img.shields.io/badge/GitHub-cloud--gaming--gpu-black?logo=github" alt="GitHub"/></a>

Biến notebook thành **máy chủ GPU đám mây hoàn chỉnh trong trình duyệt**: desktop ảo chơi game (noVNC), web terminal, dashboard realtime (GPU/CPU/RAM), treo bot Discord/Telegram/… — kết nối internet qua **Cloudflare Tunnel**, không cần domain.

## ⚡ Bắt đầu — chỉ 2 ô code

| Ô | Làm gì | Thời gian |
|---|---|---|
| **Bước 1** · Auto Setup | Cài desktop XFCE + x11vnc + noVNC + ttyd + cloudflared + aiohttp | ~2–4 phút |
| **Bước 2** · Khởi động | Chạy server, tạo 3 tunnel Cloudflare, in URL public → **giữ ô chạy** | ~30 giây |

**Yêu cầu:**
- ✅ Colab: `Runtime → Change runtime type → GPU`
- ✅ Kaggle: `Settings → Accelerator → GPU` **và** `Internet → ON`

**Sau khi Bước 2 chạy xong** bạn sẽ thấy:

```
══════════════════════════════════════════════════════════
   🎮  GPU CLOUD SERVER ĐANG CHẠY — GIỮ Ô NÀY CHẠY LIÊN TỤC!
   📊 Dashboard     https://xxx.trycloudflare.com/?key=…
   🖥️ Cloud Desktop  https://yyy.trycloudflare.com/vnc.html?…
   ⌨️ Web Terminal   https://zzz.trycloudflare.com
══════════════════════════════════════════════════════════
```

Mở **Dashboard URL** trên bất kỳ thiết bị nào → gặp giao diện quản trị đẹp: Tổng quan · Cloud Gaming · Bots · Files · Hướng dẫn.

> ⚠️ **Disclaimer:** dùng cho mục đích học tập/thử nghiệm. Chạy game / remote desktop / bot nền dài hạn có thể vi phạm ToS của Colab & Kaggle — tự chịu rủi ro tài khoản.


In [ ]:
#@title 🚀 Bước 1 · Tự động cài đặt môi trường (Desktop + VNC + Tunnel) { display-mode: "form" }
# ⚠️ Colab : Runtime → Change runtime type → GPU (T4 / L4)
# ⚠️ Kaggle: Settings → Accelerator → GPU  và  Internet → ON
import os, sys, json, time, platform, shutil, subprocess, urllib.request, pathlib

W = "─" * 56
def head(t): print(f"\n┌{W}\n  {t}\n└{W}")
def sh(cmd, timeout=None):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return p.returncode, ((p.stdout or "") + (p.stderr or ""))
def have(c): return shutil.which(c) is not None

# ── Phát hiện môi trường ──────────────────────────────────────────────
IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.exists("/kaggle")
try:
    import google.colab  # noqa
    IS_COLAB = True
except Exception:
    IS_COLAB = os.path.exists("/content") and not IS_KAGGLE
ENV_NAME = "Google Colab" if IS_COLAB else ("Kaggle" if IS_KAGGLE else "Local")

head(f"🖥️  Môi trường: {ENV_NAME}  ·  Python {platform.python_version()}  ·  {platform.machine()}")

# ── Kiểm tra GPU ──────────────────────────────────────────────────────
head("🎮 Kiểm tra GPU")
if have("nvidia-smi"):
    rc, out = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")
    print("  ✅ GPU:", out.strip().replace("\n", " | ") or "n/a")
else:
    print("  ⚠️ Không thấy NVIDIA GPU!")
    print("     • Colab : Runtime → Change runtime type → GPU (T4 / L4)")
    print("     • Kaggle: Settings → Accelerator → GPU T4 x2 / P100")
    print("  (vẫn chạy tiếp — nhưng gaming sẽ không có tăng tốc GPU)")

# ── Kiểm tra Internet ─────────────────────────────────────────────────
head("🌐 Kiểm tra Internet")
try:
    urllib.request.urlopen("https://api.github.com", timeout=10)
    print("  ✅ Online")
except Exception as e:
    print("  ❌ Offline!  Trên Kaggle hãy bật: Settings → Internet → ON, rồi chạy lại ô này.")
    raise SystemExit(1)

root = os.geteuid() == 0
SUDO = "" if (root or not have("sudo")) else "sudo -n "
LOCAL_BIN = str(pathlib.Path.home() / ".local" / "bin")
pathlib.Path(LOCAL_BIN).mkdir(parents=True, exist_ok=True)
if LOCAL_BIN not in os.environ.get("PATH", ""):
    os.environ["PATH"] += os.pathsep + LOCAL_BIN

# ── apt: desktop + vnc ────────────────────────────────────────────────
APT = ("xvfb x11vnc novnc websockify dbus-x11 x11-xserver-utils xterm xfonts-base "
       "xfce4 xfce4-terminal net-tools")
head("📦 Cài Desktop XFCE + x11vnc + noVNC (apt) — mất 1–3 phút, đừng tắt")
if have("apt-get"):
    rc, out = sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get update -y", timeout=300)
    rc2, out2 = sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends {APT}", timeout=1200)
    ok = rc2 == 0 and have("x11vnc") and have("websockify") and have("Xvfb")
    print("  ✅ Desktop + VNC OK" if ok else f"  ⚠️ apt exit={rc2} (một số gói thiếu — có thể vẫn chạy được)")
    if not ok:
        print("\n".join("    " + l for l in out2.strip().splitlines()[-12:]))
else:
    print("  ⚠️ Không có apt-get — sẽ chỉ chạy Dashboard / Bots / Terminal")

# ── pip: aiohttp ──────────────────────────────────────────────────────
head("🐍 pip: aiohttp (server dashboard)")
rc, out = sh(f"{sys.executable} -m pip install -q aiohttp", timeout=300)
print("  ✅ aiohttp OK" if rc == 0 else f"  ❌ pip lỗi:\n{out[-400:]}")

# ── cloudflared ───────────────────────────────────────────────────────
head("☁️  cloudflared (Cloudflare Tunnel — tạo URL public)")
CF = shutil.which("cloudflared")
if not CF:
    arch = {"x86_64": "amd64", "aarch64": "arm64", "armv7l": "arm"}.get(platform.machine(), "amd64")
    url = f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}"
    dst = "/usr/local/bin/cloudflared" if root else (pathlib.Path.home() / ".local/bin/cloudflared").as_posix()
    try:
        print(f"  ⬇️  tải {url}")
        urllib.request.urlretrieve(url, dst)
        os.chmod(dst, 0o755)
        CF = dst
    except Exception as e:
        print("  ❌ Tải cloudflared thất bại:", e)
if CF:
    rc, out = sh(f"{CF} --version")
    print(f"  ✅ {out.strip()}  →  {CF}")
else:
    print("  ⚠️ Không có cloudflared — Bước 2 vẫn chạy nhưng không có URL public")

# ── ttyd (web terminal) ───────────────────────────────────────────────
head("⌨️  ttyd (Web Terminal trong trình duyệt)")
if have("ttyd"):
    print("  ✅ ttyd sẵn sàng")
else:
    if have("apt-get"):
        sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y ttyd", timeout=300)
    if not have("ttyd"):
        arch = {"x86_64": "x86_64", "aarch64": "aarch64"}.get(platform.machine())
        if arch:
            url = f"https://github.com/tsl0922/ttyd/releases/download/1.7.7/ttyd.{arch}"
            dst = "/usr/local/bin/ttyd" if root else (pathlib.Path.home() / ".local/bin/ttyd").as_posix()
            try:
                print(f"  ⬇️  tải {url}")
                urllib.request.urlretrieve(url, dst)
                os.chmod(dst, 0o755)
            except Exception as e:
                print("  ❌ Tải ttyd thất bại:", e)
    print("  ✅ ttyd OK" if have("ttyd") else "  ⚠️ ttyd thiếu — Web Terminal sẽ bị tắt")

# ── thư mục làm việc ──────────────────────────────────────────────────
HOME = pathlib.Path.home()
for d in ("gpu_server", "bots", "bot_logs"):
    (HOME / d).mkdir(exist_ok=True)
pathlib.Path("/tmp/tunnels.json").write_text("{}")

head("✅ BƯỚC 1 HOÀN TẤT — giờ chạy ô 『Bước 2 · Khởi động Server + Tunnel』")
print("  💡 Giữ notebook đang chạy (không đóng tab) để server luôn online.")


In [ ]:
%%writefile dashboard.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# ═══════════════════════════════════════════════════════════════════
#  🎮 GPU CLOUD SERVER · Dashboard Web (aiohttp, không cần framework khác)
#  Tab: Tổng quan · Cloud Gaming · Bots · Files · Hướng dẫn
#  Chạy được trên Google Colab / Kaggle / máy Linux bất kỳ.
#  Bảo mật bằng Access Key (biến môi trường DASH_TOKEN).
# ═══════════════════════════════════════════════════════════════════
import asyncio
import json
import os
import pathlib
import re
import shutil
import socket
import subprocess
import sys
import threading
import time
from aiohttp import web

HOME = pathlib.Path.home()
BOTS_DIR = HOME / "bots"; BOTS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR = HOME / "bot_logs"; LOGS_DIR.mkdir(parents=True, exist_ok=True)

PORT = int(os.environ.get("DASH_PORT", "8080"))
TOKEN = os.environ.get("DASH_TOKEN") or ""
AUTH_ENABLED = bool(os.environ.get("DASH_TOKEN"))
TUNNELS_FILE = pathlib.Path(os.environ.get("TUNNELS_FILE", "/tmp/tunnels.json"))

BOTS = {}          # name -> {"proc": Popen, "started": ts, "log": path, "fh": file}
ACTIONS = {}       # id  -> {"cmd","out","done","rc","t"}
GDIR = pathlib.Path(os.environ.get("GSERVER_DIR", str(HOME / "gpu_server")))

# ─────────────────────────── helpers ───────────────────────────
def detect_env():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.path.exists("/kaggle"):
        return "Kaggle"
    try:
        import google.colab  # noqa
        return "Google Colab"
    except Exception:
        if os.path.exists("/content"):
            return "Google Colab"
    return "Local"

def human_time(s):
    s = int(s); h, r = divmod(s, 3600); m, _ = divmod(r, 60)
    return (f"{h}h {m}m" if h else f"{m}m {s % 60}s")

def _read_cpu():
    with open("/proc/stat") as f:
        parts = f.readline().split()
    vals = list(map(int, parts[1:8]))
    idle = vals[3] + (vals[4] if len(vals) > 4 else 0)
    return sum(vals), idle

_cpu_prev = None
def cpu_percent():
    global _cpu_prev
    if _cpu_prev is None:
        _cpu_prev = _read_cpu()
        time.sleep(0.15)
    cur = _read_cpu()
    dt = cur[0] - _cpu_prev[0]
    di = cur[1] - _cpu_prev[1]
    _cpu_prev = cur
    return round(100 * (1 - di / max(dt, 1)), 1)

def mem_info():
    d = {}
    for line in open("/proc/meminfo"):
        k, v = line.split(":")
        d[k.strip()] = int(v.strip().split()[0])
    total = d.get("MemTotal", 0); avail = d.get("MemAvailable", total)
    used = total - avail
    return {"total_gb": round(total / 1048576, 2),
            "used_gb": round(used / 1048576, 2),
            "percent": round(100 * used / max(total, 1), 1)}

def gpu_info():
    if not shutil.which("nvidia-smi"):
        return None
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total,temperature.gpu",
             "--format=csv,noheader,nounits"], text=True, timeout=5)
        name, util, mu, mt, temp = [x.strip() for x in out.strip().splitlines()[0].split(",")]
        return {"name": name, "util": int(util or 0),
                "mem_used_mb": int(mu or 0), "mem_total_mb": int(mt or 0),
                "temp": int(temp or 0)}
    except Exception:
        return None

def read_tunnels():
    try:
        return json.loads(TUNNELS_FILE.read_text())
    except Exception:
        return {}

# ─────────────────────────── services status ───────────────────────────
def _port_open(port, timeout=0.4):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=timeout):
            return True
    except OSError:
        return False

def _proc_running(needle):
    me = os.getpid()
    for d in pathlib.Path("/proc").iterdir():
        if not d.name.isdigit() or int(d.name) == me:
            continue
        try:
            cmd = (d / "cmdline").read_bytes().decode("utf-8", "ignore").replace("\x00", " ")
        except Exception:
            continue
        if needle in cmd:
            return True
    return False

def _log_tail(fname, lines=12):
    f = GDIR / fname
    if not f.exists():
        return []
    try:
        return f.read_bytes()[-4096:].decode("utf-8", "ignore").splitlines()[-lines:]
    except Exception:
        return []

SERVICE_DEFS = [
    ("xvfb",       "🖥️ Desktop ảo (Xvfb)",       "Xvfb",       None, "xvfb.log"),
    ("xfce",       "🪟 XFCE session",             "xfce",       None, "xfce4.log"),
    ("x11vnc",     "🔗 VNC (x11vnc)",             "x11vnc",     5901, "x11vnc.log"),
    ("websockify", "🌐 noVNC (websockify)",       "websockify", 6080, "websockify.log"),
    ("ttyd",       "⌨️ Terminal (ttyd)",          "ttyd",       7681, "ttyd.log"),
    ("cloudflared","☁️ Tunnel (cloudflared)",     "cloudflared", None, None),
]

def services_status():
    out = []
    for key, name, proc, port, logf in SERVICE_DEFS:
        out.append({
            "key": key, "name": name,
            "running": _proc_running(proc),
            "port": port,
            "port_open": _port_open(port) if port else None,
            "log": _log_tail(logf) if logf else [],
        })
    return out

async def api_services(request):
    svcs = await asyncio.get_running_loop().run_in_executor(None, services_status)
    return web.json_response({"services": svcs})

# ─────────────────────────── auth ───────────────────────────
@web.middleware
async def auth_mw(request, handler):
    if AUTH_ENABLED:
        key = request.query.get("key") or request.cookies.get("gpu_key") or request.headers.get("X-Key")
        if key != TOKEN:
            return web.json_response({"error": "unauthorized — thêm ?key=<ACCESS_KEY> vào URL"}, status=401)
    resp = await handler(request)
    if AUTH_ENABLED:
        try:
            resp.set_cookie("gpu_key", TOKEN, httponly=True, samesite="Lax", max_age=86400 * 7)
        except Exception:
            pass
    return resp

# ─────────────────────────── API: stats / tunnels ───────────────────────────
async def api_stats(request):
    disk = shutil.disk_usage(HOME)
    running = sum(1 for b in BOTS.values() if b["proc"].poll() is None)
    return web.json_response({
        "cpu": cpu_percent(),
        "load": [round(x, 2) for x in os.getloadavg()],
        "mem": mem_info(),
        "gpu": await asyncio.get_running_loop().run_in_executor(None, gpu_info),
        "disk": {"total_gb": round(disk.total / 1e9, 1),
                 "used_gb": round(disk.used / 1e9, 1),
                 "percent": round(100 * disk.used / max(disk.total, 1), 1)},
        "uptime_h": human_time(float(open("/proc/uptime").read().split()[0]) if os.path.exists("/proc/uptime") else 0),
        "python": sys.version.split()[0],
        "env": detect_env(),
        "bots_running": running,
    })

async def api_tunnels(request):
    t = read_tunnels()
    return web.json_response({
        "tunnels": t,
        "local": {"dashboard": f"http://localhost:{PORT}",
                  "desktop": "http://localhost:6080",
                  "terminal": "http://localhost:7681"},
        "auth": AUTH_ENABLED,
    })

# ─────────────────────────── API: bots ───────────────────────────
def bot_status(name):
    b = BOTS.get(name)
    if not b:
        return "stopped"
    rc = b["proc"].poll()
    if rc is None:
        return "running"
    return f"exited({rc})"

async def api_bots(request):
    items = []
    for p in sorted(BOTS_DIR.glob("*.py")):
        name = p.stem
        b = BOTS.get(name)
        alive = b is not None and b["proc"].poll() is None
        items.append({
            "name": name,
            "status": bot_status(name),
            "pid": b["proc"].pid if alive else None,
            "uptime": human_time(time.time() - b["started"]) if alive else "",
            "size": p.stat().st_size,
            "has_env": (BOTS_DIR / f"{name}.env").exists(),
        })
    return web.json_response({"bots": items})

async def api_bots_upload(request):
    reader = await request.multipart()
    name, env_text, saved_name = None, "", None
    while True:
        part = await reader.next()
        if part is None:
            break
        if part.name == "file":
            fname = pathlib.Path(part.filename or "bot.py").name
            if not fname.endswith(".py"):
                fname += ".py"
            data = b""
            while True:
                chunk = await part.read_chunk(65536)
                if not chunk:
                    break
                data += chunk
            (BOTS_DIR / fname).write_bytes(data)
            saved_name = fname[:-3]
            if not name:
                name = saved_name
        elif part.name == "name":
            requested = (await part.read()).decode().strip()
            if requested:
                name = re.sub(r"[^A-Za-z0-9_.-]", "_", requested)[:64]
        elif part.name == "env":
            env_text = (await part.read()).decode()
    if not saved_name or not name:
        return web.json_response({"error": "thiếu file .py"}, status=400)
    src = BOTS_DIR / f"{saved_name}.py"
    dst = BOTS_DIR / f"{name}.py"
    if src != dst:
        src.rename(dst)
    if env_text.strip():
        (BOTS_DIR / f"{name}.env").write_text(env_text)
    return web.json_response({"ok": True, "name": name})

async def api_bot_start(request):
    name = request.match_info["name"]
    script = BOTS_DIR / f"{name}.py"
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", name) or not script.exists():
        return web.json_response({"error": "bot không tồn tại"}, status=404)
    if name in BOTS and BOTS[name]["proc"].poll() is None:
        return web.json_response({"ok": True, "already": True})
    env = dict(os.environ)
    envf = BOTS_DIR / f"{name}.env"
    if envf.exists():
        for line in envf.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    logf = LOGS_DIR / f"{name}.log"
    fh = open(logf, "ab", buffering=0)
    fh.write(f"\n══════ ▶ start {time.strftime('%Y-%m-%d %H:%M:%S')} ══════\n".encode())
    proc = subprocess.Popen([sys.executable, str(script)], stdout=fh, stderr=subprocess.STDOUT,
                            stdin=subprocess.DEVNULL, env=env, cwd=str(BOTS_DIR),
                            start_new_session=True)
    BOTS[name] = {"proc": proc, "started": time.time(), "log": logf, "fh": fh}
    return web.json_response({"ok": True, "pid": proc.pid})

async def api_bot_stop(request):
    name = request.match_info["name"]
    b = BOTS.get(name)
    if not b or b["proc"].poll() is not None:
        return web.json_response({"ok": True, "note": "không đang chạy"})
    b["proc"].terminate()
    try:
        b["proc"].wait(timeout=4)
    except Exception:
        b["proc"].kill()
    try:
        b["fh"].close()
    except Exception:
        pass
    return web.json_response({"ok": True})

async def api_bot_logs(request):
    name = request.match_info["name"]
    lines = int(request.query.get("lines", "200"))
    logf = LOGS_DIR / f"{name}.log"
    data = []
    if logf.exists():
        raw = logf.read_bytes()[-16384:].decode("utf-8", "replace")
        data = raw.splitlines()[-lines:]
    return web.json_response({"lines": data, "running": bot_status(name) == "running"})

async def api_bot_delete(request):
    name = request.match_info["name"]
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", name):
        return web.json_response({"error": "tên không hợp lệ"}, status=400)
    b = BOTS.pop(name, None)
    if b and b["proc"].poll() is None:
        b["proc"].terminate()
    for f in (BOTS_DIR / f"{name}.py", BOTS_DIR / f"{name}.env", LOGS_DIR / f"{name}.log"):
        try:
            f.unlink()
        except Exception:
            pass
    return web.json_response({"ok": True})

# ─────────────────────────── API: files ───────────────────────────
def safe_join(rel):
    home = HOME.resolve()
    if not rel:
        return home
    p = pathlib.Path(rel)
    q = p.resolve() if p.is_absolute() else (home / p).resolve()
    try:
        q.relative_to(home)
    except ValueError:
        raise web.HTTPForbidden(text="Đường dẫn nằm ngoài thư mục home")
    return q

async def api_files(request):
    p = safe_join(request.query.get("path", "."))
    items = []
    for c in sorted(p.iterdir(), key=lambda x: (x.is_file(), x.name.lower())):
        if c.name.startswith("."):
            continue
        try:
            st = c.stat()
            items.append({"name": c.name, "is_dir": c.is_dir(), "size": st.st_size})
        except Exception:
            pass
    rel = str(p.relative_to(HOME))
    return web.json_response({"path": "" if rel == "." else rel, "items": items})

async def api_files_download(request):
    p = safe_join(request.query.get("path", ""))
    if not p.exists():
        return web.json_response({"error": "file không tồn tại"}, status=404)
    if p.is_dir():
        return web.json_response({"error": "là thư mục — hãy nén trước"}, status=400)
    return web.FileResponse(p, headers={"Content-Disposition": f'attachment; filename="{p.name}"'})

async def api_files_upload(request):
    reader = await request.multipart()
    dest, saved = ".", None
    while True:
        part = await reader.next()
        if part is None:
            break
        if part.name == "path":
            dest = (await part.read()).decode().strip()
        elif part.name == "file":
            fname = pathlib.Path(part.filename or "file.bin").name
            data = b""
            while True:
                chunk = await part.read_chunk(65536)
                if not chunk:
                    break
                data += chunk
            saved = (fname, data)
    if not saved:
        return web.json_response({"error": "thiếu file"}, status=400)
    d = safe_join(dest)
    d.mkdir(parents=True, exist_ok=True)
    (d / saved[0]).write_bytes(saved[1])
    return web.json_response({"ok": True})

async def api_files_delete(request):
    body = await request.json()
    p = safe_join(body.get("path", ""))
    if p == HOME.resolve():
        return web.json_response({"error": "không thể xóa home"}, status=400)
    if p.is_dir():
        shutil.rmtree(p)
    else:
        p.unlink(missing_ok=True)
    return web.json_response({"ok": True})

async def api_files_mkdir(request):
    body = await request.json()
    p = safe_join(body.get("path", ""))
    p.mkdir(parents=True, exist_ok=True)
    return web.json_response({"ok": True})

# ─────────────────────────── API: actions (quick commands) ───────────────────────────
WHITELIST = {
    "install_steam": ("dpkg --add-architecture i386 && apt-get update -y && "
                      "DEBIAN_FRONTEND=noninteractive apt-get install -y steam || "
                      "DEBIAN_FRONTEND=noninteractive apt-get install -y steam-installer"),
    "install_wine": ("dpkg --add-architecture i386 && apt-get update -y && "
                     "DEBIAN_FRONTEND=noninteractive apt-get install -y wine wine64 winbind || "
                     "DEBIAN_FRONTEND=noninteractive apt-get install -y wine"),
    "install_discord_py": "python3 -m pip install -U discord.py",
    "install_telegram": "python3 -m pip install -U pyTelegramBotAPI python-telegram-bot",
    "install_firefox": "DEBIAN_FRONTEND=noninteractive apt-get install -y firefox || "
                       "DEBIAN_FRONTEND=noninteractive apt-get install -y firefox-esr chromium-browser",
}

async def api_action(request):
    body = await request.json()
    act = body.get("action")
    cmd = WHITELIST.get(act) or (body.get("cmd") or "").strip()
    if not cmd:
        return web.json_response({"error": "thiếu action/cmd"}, status=400)
    aid = f"{int(time.time() * 1000)}"
    ACTIONS[aid] = {"id": aid, "cmd": cmd, "out": [], "done": False, "rc": None,
                    "t": time.strftime("%H:%M:%S")}
    def worker():
        try:
            env = dict(os.environ)
            env.setdefault("DISPLAY", ":1")
            env["DEBIAN_FRONTEND"] = "noninteractive"
            proc = subprocess.Popen(["bash", "-lc", cmd], stdout=subprocess.PIPE,
                                    stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
            for line in proc.stdout:
                a = ACTIONS.get(aid)
                if a is not None:
                    a["out"].append(line.rstrip("\n"))
                    a["out"] = a["out"][-400:]
            ACTIONS[aid]["rc"] = proc.wait()
        except Exception as e:
            ACTIONS[aid]["out"].append(f"error: {e}")
        ACTIONS[aid]["done"] = True
    threading.Thread(target=worker, daemon=True).start()
    return web.json_response({"ok": True, "id": aid})

async def api_action_logs(request):
    out = []
    for aid in sorted(ACTIONS, reverse=True)[:12]:
        a = ACTIONS[aid]
        out.append({"id": aid, "cmd": a["cmd"], "t": a["t"], "done": a["done"],
                    "rc": a["rc"], "out": a["out"][-200:]})
    return web.json_response({"actions": out})

# ─────────────────────────── HTML ───────────────────────────
DASH_HTML = r'''<!doctype html>
<html lang="vi"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>🎮 GPU Cloud Server</title>
<style>
:root{--bg:#0a0714;--card:rgba(255,255,255,.045);--line:rgba(255,255,255,.09);--tx:#e9e6f5;--mut:#9b94b8;--acc:#8b5cf6;--acc2:#22d3ee;--ok:#34d399;--warn:#fbbf24;--bad:#f87171}
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:ui-sans-serif,system-ui,'Segoe UI',Roboto,sans-serif;background:radial-gradient(1100px 600px at 85% -10%,rgba(139,92,246,.20),transparent),radial-gradient(900px 500px at -10% 110%,rgba(34,211,238,.12),transparent),var(--bg);color:var(--tx);min-height:100vh}
header{position:sticky;top:0;z-index:10;display:flex;align-items:center;gap:14px;padding:12px 20px;background:rgba(10,7,20,.8);backdrop-filter:blur(10px);border-bottom:1px solid var(--line);flex-wrap:wrap}
.logo{font-weight:800;font-size:17px;background:linear-gradient(90deg,#a78bfa,#22d3ee);-webkit-background-clip:text;background-clip:text;color:transparent;white-space:nowrap}
nav{display:flex;gap:6px;flex-wrap:wrap}
.tab-btn{padding:8px 14px;border-radius:10px;border:1px solid transparent;background:transparent;color:var(--mut);cursor:pointer;font-size:14px;font-weight:600}
.tab-btn.on{color:#fff;background:rgba(139,92,246,.2);border-color:rgba(139,92,246,.45)}
.pill{margin-left:auto;font-size:12px;color:var(--mut);border:1px dashed var(--line);padding:6px 10px;border-radius:999px}
main{max-width:1100px;margin:0 auto;padding:22px 16px 70px}
section.tab{display:none}section.tab.on{display:block}
h2{font-size:19px;margin:4px 0 14px}
.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(215px,1fr));gap:14px}
.card{background:var(--card);border:1px solid var(--line);border-radius:16px;padding:16px;margin-bottom:14px}
.card h3{font-size:13px;color:var(--mut);font-weight:600;margin-bottom:8px}
.big{font-size:20px;font-weight:800}
.bar{height:8px;border-radius:99px;background:rgba(255,255,255,.08);overflow:hidden;margin-top:8px}
.bar i{display:block;height:100%;border-radius:99px;background:linear-gradient(90deg,var(--acc),var(--acc2));transition:width .4s}
.btn{display:inline-flex;align-items:center;gap:8px;padding:9px 15px;border-radius:12px;border:1px solid var(--line);background:rgba(255,255,255,.05);color:var(--tx);cursor:pointer;font-size:14px;font-weight:600;text-decoration:none}
.btn:hover{border-color:var(--acc)}
.btn.primary{background:linear-gradient(90deg,#7c3aed,#6d28d9);border-color:#8b5cf6}
.btn.cy{background:linear-gradient(90deg,#0891b2,#0e7490);border-color:#22d3ee}
.btn.danger{border-color:rgba(248,113,113,.4);color:#fca5a5}
table{width:100%;border-collapse:collapse;font-size:14px}
th,td{text-align:left;padding:9px 10px;border-bottom:1px solid var(--line)}
th{color:var(--mut);font-size:12px;text-transform:uppercase;letter-spacing:.05em}
input,textarea{background:rgba(255,255,255,.06);border:1px solid var(--line);border-radius:10px;color:var(--tx);padding:9px 12px;font-size:14px;width:100%}
textarea{min-height:90px;font-family:ui-monospace,monospace}
pre{background:#0d0a1a;border:1px solid var(--line);border-radius:12px;padding:12px;font-size:12.5px;max-height:320px;overflow:auto;white-space:pre-wrap;font-family:ui-monospace,Menlo,Consolas,monospace;color:#c9c4e2}
.url{font-family:ui-monospace,monospace;font-size:13px;word-break:break-all;color:#a5f3fc}
.row{display:flex;gap:10px;flex-wrap:wrap;align-items:center}
.mut{color:var(--mut);font-size:13px}
.tag{font-size:11px;padding:3px 9px;border-radius:99px;border:1px solid;white-space:nowrap}
.tag.ok{color:var(--ok);border-color:rgba(52,211,153,.4)}.tag.bad{color:var(--bad);border-color:rgba(248,113,113,.4)}.tag.warn{color:var(--warn);border-color:rgba(251,191,36,.4)}
a.flink{color:#a5f3fc;text-decoration:none}a.flink:hover{text-decoration:underline}
#modal{display:none;position:fixed;inset:0;background:rgba(0,0,0,.65);z-index:50;align-items:center;justify-content:center}
#modal>div{width:min(760px,94vw);max-height:82vh;display:flex;flex-direction:column;gap:10px;background:#120c22;border:1px solid var(--line);border-radius:16px;padding:16px}
.toast{position:fixed;bottom:18px;left:50%;transform:translateX(-50%);background:#1e1633;border:1px solid var(--acc);padding:10px 18px;border-radius:12px;font-size:14px;opacity:0;transition:.3s;z-index:99;pointer-events:none}
.toast.show{opacity:1}
.hero{background:linear-gradient(120deg,rgba(124,58,237,.25),rgba(34,211,238,.12));border:1px solid rgba(139,92,246,.35);border-radius:18px;padding:22px}
</style></head>
<body>
<header>
  <div class="logo">🎮 GPU CLOUD SERVER</div>
  <nav>
    <button class="tab-btn on" data-tab="overview">📊 Tổng quan</button>
    <button class="tab-btn" data-tab="gaming">🎮 Cloud Gaming</button>
    <button class="tab-btn" data-tab="bots">🤖 Bots</button>
    <button class="tab-btn" data-tab="files">📁 Files</button>
    <button class="tab-btn" data-tab="help">❓ Hướng dẫn</button>
  </nav>
  <div class="pill" id="authPill">🔐 protected</div>
</header>
<main>

<section class="tab on" id="tab-overview">
  <h2>📊 Tổng quan hệ thống</h2>
  <div class="grid">
    <div class="card"><h3>🎮 GPU</h3><div id="stGPU" class="big">…</div>
      <div class="mut" id="gpuUtilTxt">util —</div><div class="bar" id="gpuUtilBar"><i style="width:0%"></i></div>
      <div class="mut" id="gpuMemTxt">vram —</div><div class="bar" id="gpuMemBar"><i style="width:0%"></i></div>
      <div class="mut" id="gpuTemp"></div></div>
    <div class="card"><h3>🧠 CPU</h3><div class="big" id="cpuTxt">—</div><div class="bar" id="cpuBar"><i style="width:0%"></i></div></div>
    <div class="card"><h3>💾 RAM</h3><div class="big" id="ramTxt">—</div><div class="bar" id="ramBar"><i style="width:0%"></i></div></div>
    <div class="card"><h3>🗄️ Disk</h3><div class="big" id="diskTxt">—</div><div class="bar" id="diskBar"><i style="width:0%"></i></div></div>
  </div>
  <div class="card"><div class="mut" id="stMeta">…</div></div>
  <div class="card"><h3>🌐 Public URLs (Cloudflare Tunnel)</h3>
    <table><thead><tr><th>Dịch vụ</th><th>URL</th><th></th></tr></thead><tbody id="tunnelRows"></tbody></table>
    <p class="mut" style="margin-top:8px">💡 URL tunnel là tạm thời — mỗi lần chạy lại notebook sẽ có URL mới.</p>
  </div>
  <div class="card"><h3>⚙️ Dịch vụ nền</h3>
    <table><thead><tr><th>Dịch vụ</th><th>Trạng thái</th><th>Cổng</th><th></th></tr></thead>
    <tbody id="svcRows"></tbody></table>
    <p class="mut" style="margin-top:8px">❌ = chưa chạy. Nếu Cloud Desktop không vào được, bấm 📜 xem log từng dịch vụ để tìm nguyên nhân.</p>
  </div>
</section>

<section class="tab" id="tab-gaming">
  <div class="hero">
    <h2>🎮 Cloud Gaming — Desktop ảo trong trình duyệt</h2>
    <p class="mut">Desktop XFCE chạy trên GPU VM, xem/điều khiển qua noVNC. Cài Steam/Wine rồi mở game ngay trong desktop.</p>
    <div class="row" style="margin-top:14px">
      <button class="btn primary" id="btnDesktop">🖥️ Mở Cloud Desktop</button>
      <button class="btn cy" id="btnTerminal">⌨️ Mở Web Terminal</button>
      <span class="mut" id="desktopState"></span>
    </div>
  </div>
  <div class="card">
    <h3>⚡ Hành động nhanh</h3>
    <div class="row">
      <button class="btn" onclick="doAction('install_steam')">🎮 Cài Steam</button>
      <button class="btn" onclick="doAction('install_wine')">🍷 Cài Wine</button>
      <button class="btn" onclick="doAction('install_firefox')">🦊 Cài Firefox/Chromium</button>
      <button class="btn" onclick="doAction('install_discord_py')">📦 pip discord.py</button>
      <button class="btn" onclick="doAction('install_telegram')">📦 pip telegram</button>
    </div>
    <form id="cmdForm" class="row" style="margin-top:12px">
      <input id="cmdInput" placeholder="Lệnh tuỳ ý, ví dụ: pip install -U yt-dlp   ·   DISPLAY=:1 google-chrome" style="flex:1;min-width:260px">
      <button class="btn primary" type="submit">▶ Chạy</button>
    </form>
    <pre id="actionLog" style="margin-top:12px">(log hành động…)</pre>
  </div>
  <div class="card"><h3>📈 Mẹo gaming mượt hơn</h3>
    <ul class="mut" style="line-height:1.9">
      <li>Chơi game nhẹ / game cũ / emulator — noVNC truyền ảnh qua mạng nên game AAA sẽ lag.</li>
      <li>Trong desktop, giảm độ phân giải xuống 1280×720 sẽ mượt hơn đáng kể.</li>
      <li>Cài game bằng Steam trong Desktop (tab này chỉ cài phần mềm, mở game trong Desktop).</li>
      <li>Audio không truyền qua noVNC — dùng game không cần âm thanh hoặc cắm tai nghe tâm lý 🙂.</li>
    </ul>
  </div>
</section>

<section class="tab" id="tab-bots">
  <h2>🤖 Treo bot (Discord / Telegram / …)</h2>
  <div class="card">
    <table><thead><tr><th>Bot</th><th>Trạng thái</th><th>Uptime</th><th>Env</th><th>Hành động</th></tr></thead>
    <tbody id="botRows"></tbody></table>
  </div>
  <div class="card">
    <h3>⬆️ Upload bot mới (.py + biến môi trường)</h3>
    <form id="botForm">
      <div class="row">
        <input type="file" id="botFile" name="file" accept=".py" required style="flex:1;min-width:220px">
        <input name="name" placeholder="Tên bot (tuỳ chọn)" style="flex:1;min-width:180px">
      </div>
      <textarea name="env" placeholder="Biến môi trường, mỗi dòng KEY=VALUE — ví dụ:&#10;DISCORD_TOKEN=xxxx&#10;TELEGRAM_TOKEN=xxxx"></textarea>
      <div class="row" style="margin-top:10px">
        <button class="btn primary" type="submit">💾 Lưu bot</button>
        <span class="mut">Sau khi lưu, bấm ▶ Start để chạy. Token nằm trong file .env, không in ra log.</span>
      </div>
    </form>
  </div>
</section>

<section class="tab" id="tab-files">
  <h2>📁 Quản lý file</h2>
  <div class="card">
    <div class="row">
      <input id="filePath" placeholder="đường dẫn tương đối từ home" style="flex:1;min-width:220px">
      <button class="btn" id="fileGo">Go</button>
      <button class="btn" id="btnMkdir">➕ Thư mục</button>
      <button class="btn primary" id="btnUpFile">⬆️ Upload</button>
      <input type="file" id="fileInput" style="display:none">
    </div>
    <table style="margin-top:12px"><thead><tr><th>Tên</th><th>Kích thước</th><th></th></tr></thead><tbody id="fileRows"></tbody></table>
    <p class="mut" style="margin-top:8px">Home = thư mục user của VM (Colab: /root, Kaggle: /root). File bot nằm trong ~/bots.</p>
  </div>
</section>

<section class="tab" id="tab-help">
  <h2>❓ Hướng dẫn nhanh</h2>
  <div class="card"><ul class="mut" style="line-height:2">
    <li><b>Cloud Gaming:</b> tab 🎮 → Mở Cloud Desktop → trong desktop cài Steam (bấm nút Cài Steam trước) → đăng nhập → chơi game. Điều khiển bằng chuột/bàn phím của bạn.</li>
    <li><b>Treo bot:</b> tab 🤖 → upload file .py + đặt token vào ô Env (KEY=VALUE) → Start. Xem log realtime bằng nút 📜.</li>
    <li><b>Web Terminal:</b> tab 🎮 → Mở Web Terminal — bash đầy đủ trên VM (cài gì cũng được).</li>
    <li><b>Files:</b> tab 📁 — upload/download/xoá file trong VM.</li>
    <li><b>Bảo mật:</b> Dashboard yêu cầu access key trong URL (?key=…). Desktop/Terminal dùng tunnel riêng không có key — đừng chia sẻ URL công khai.</li>
    <li><b>Giữ server sống:</b> không đóng tab notebook; Colab/Kaggle sẽ ngắt nếu bạn đóng hoặc quá idle quá lâu.</li>
  </ul></div>
</section>

</main>

<div id="modal"><div>
  <div class="row"><b id="modalTitle">Logs</b><button class="btn" id="modalClose" style="margin-left:auto">Đóng ✕</button></div>
  <pre id="modalPre" style="max-height:56vh"></pre>
</div></div>
<div class="toast" id="toast"></div>

<script>
const KEY = "__TOKEN__";
const $ = s => document.querySelector(s);
const $$ = s => [...document.querySelectorAll(s)];
function toast(m){const t=$("#toast");t.textContent=m;t.classList.add("show");setTimeout(()=>t.classList.remove("show"),2400);}
async function api(path, opts={}){
  const u = new URL(path, location.href);
  if(KEY) u.searchParams.set("key", KEY);
  const r = await fetch(u, opts);
  if(r.status===401){toast("❌ Access key sai — mở URL kèm ?key=…");throw new Error("401");}
  if(!r.ok){let e={};try{e=await r.json()}catch(_){};throw new Error(e.error||("HTTP "+r.status));}
  return r.json();
}
function openURL(u){ if(u && !String(u).startsWith("(")) window.open(u,"_blank"); else toast("Tunnel chưa sẵn sàng — đợi vài giây rồi thử lại"); }
function copyText(t){ navigator.clipboard.writeText(t).then(()=>toast("📋 Đã copy")); }
$$(".tab-btn").forEach(b=>b.onclick=()=>{
  $$(".tab-btn").forEach(x=>x.classList.toggle("on",x===b));
  $$("section.tab").forEach(s=>s.classList.toggle("on",s.id==="tab-"+b.dataset.tab));
});

/* ── overview ── */
async function loadStats(){
  try{
    const s = await api("/api/stats");
    const g = s.gpu;
    $("#stGPU").innerHTML = g ? `<div class="big">${g.name}</div>` : `<div class="big">Không có GPU</div>`;
    if(g){
      $("#gpuUtilBar").firstElementChild.style.width = g.util+"%";
      $("#gpuUtilTxt").textContent = "⚙️ util "+g.util+"%";
      $("#gpuMemBar").firstElementChild.style.width = Math.round(100*g.mem_used_mb/Math.max(g.mem_total_mb,1))+"%";
      $("#gpuMemTxt").textContent = `🧬 VRAM ${g.mem_used_mb} / ${g.mem_total_mb} MB`;
      $("#gpuTemp").textContent = "🌡️ "+g.temp+"°C";
    }
    $("#cpuBar").firstElementChild.style.width = s.cpu+"%";
    $("#cpuTxt").textContent = s.cpu+"%  ·  load "+s.load.join(" / ");
    $("#ramBar").firstElementChild.style.width = s.mem.percent+"%";
    $("#ramTxt").textContent = `${s.mem.used_gb} / ${s.mem.total_gb} GB (${s.mem.percent}%)`;
    $("#diskBar").firstElementChild.style.width = s.disk.percent+"%";
    $("#diskTxt").textContent = `${s.disk.used_gb} / ${s.disk.total_gb} GB (${s.disk.percent}%)`;
    $("#stMeta").textContent = `⏱️ uptime ${s.uptime_h}   ·   🐍 Python ${s.python}   ·   ☁️ ${s.env}   ·   🤖 ${s.bots_running} bot đang chạy`;
  }catch(e){}
}
let T = {};
async function loadTunnels(){
  try{
    T = (await api("/api/tunnels")).tunnels;
    const rows = [
      ["📊 Dashboard","dashboard", u=>u+"/?key="+KEY],
      ["🖥️ Cloud Desktop","desktop", u=>u+"/vnc.html?autoconnect=1&resize=scale&reconnect=1"],
      ["⌨️ Web Terminal","terminal", u=>u],
    ];
    $("#tunnelRows").innerHTML = rows.map(([lbl,k,fn])=>{
      const u = T[k];
      return `<tr><td>${lbl}</td><td class="url">${u?fn(u):"<span class='tag warn'>đang kết nối…</span>"}</td>
      <td>${u?`<button class="btn" onclick='copyText("${fn(u)}")'>📋</button> <button class="btn cy" onclick='openURL("${fn(u)}")'>↗</button>`:"—"}</td></tr>`;
    }).join("");
    $("#desktopState").textContent = T.desktop ? "✅ Desktop sẵn sàng" : "⏳ Desktop chưa có tunnel…";
    $("#authPill").textContent = KEY ? "🔐 key: "+KEY.slice(0,4)+"…" : "🔓 local mode";
  }catch(e){}
}
$("#btnDesktop").onclick=()=>openURL(T.desktop ? T.desktop+"/vnc.html?autoconnect=1&resize=scale&reconnect=1" : null);
$("#btnTerminal").onclick=()=>openURL(T.terminal || null);

/* ── services ── */
let SVC={};
async function loadServices(){
  try{
    const {services}=await api("/api/services");
    $("#svcRows").innerHTML=services.map(s=>{
      const st=s.running?"<span class='tag ok'>✅ chạy</span>":"<span class='tag bad'>❌ tắt</span>";
      const pt=s.port==null?"—":(s.port_open?"🟢 mở "+s.port:"🔴 đóng");
      return `<tr><td>${s.name}</td><td>${st}</td><td>${pt}</td><td>${s.log&&s.log.length?`<button class="btn" onclick="svcLog('${s.key}')">📜</button>`:""}</td></tr>`;
    }).join("");
    services.forEach(s=>SVC[s.key]=s.log);
  }catch(e){}
}
function svcLog(k){
  $("#modal").style.display="flex"; $("#modalTitle").textContent="📜 Log · "+k;
  $("#modalPre").textContent=(SVC[k]||[]).join("\n")||"(log trống)";
  clearInterval(logTimer);
}

/* ── gaming actions ── */
async function doAction(name){
  await api("/api/action",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({action:name})});
  toast("⚙️ Đã gửi — xem log bên dưới"); pollActions();
}
$("#cmdForm").onsubmit = async ev=>{
  ev.preventDefault();
  const c = $("#cmdInput").value.trim(); if(!c) return;
  await api("/api/action",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({cmd:c})});
  $("#cmdInput").value=""; pollActions();
};
async function pollActions(){
  try{
    const {actions} = await api("/api/action/logs");
    $("#actionLog").textContent = actions.map(a=>
      `【${a.t}】$ ${a.cmd}\n${(a.out||[]).join("\n")}${a.done?`\n— exit ${a.rc} —`:"\n… đang chạy …"}`).join("\n\n")
      || "(chưa có lệnh nào được chạy)";
    const p=$("#actionLog"); p.scrollTop=p.scrollHeight;
  }catch(e){}
}

/* ── bots ── */
async function loadBots(){
  try{
    const {bots} = await api("/api/bots");
    $("#botRows").innerHTML = bots.map(b=>`<tr>
      <td><b>${b.name}</b></td>
      <td><span class="tag ${b.status==="running"?"ok":(b.status.startsWith("exited")?"warn":"bad")}">${b.status}</span></td>
      <td>${b.uptime||"—"}</td>
      <td>${b.has_env?"🔑 có":"—"}</td>
      <td>${b.status==="running"
        ? `<button class="btn danger" onclick="botStop('${b.name}')">■ Stop</button>`
        : `<button class="btn primary" onclick="botStart('${b.name}')">▶ Start</button>`}
        <button class="btn" onclick="botLogs('${b.name}')">📜</button>
        <button class="btn danger" onclick="botDelete('${b.name}')">🗑️</button></td></tr>`).join("")
      || `<tr><td colspan="5" class="mut">Chưa có bot nào — upload ở form bên dưới 🙂</td></tr>`;
  }catch(e){}
}
async function botStart(n){try{await api(`/api/bots/${n}/start`,{method:"POST"});toast("▶ Đã start "+n);loadBots();}catch(e){toast("❌ "+e.message);}}
async function botStop(n){try{await api(`/api/bots/${n}/stop`,{method:"POST"});toast("■ Đã stop "+n);loadBots();}catch(e){toast("❌ "+e.message);}}
async function botDelete(n){if(!confirm("Xóa bot "+n+"?"))return;await api(`/api/bots/${n}/delete`,{method:"POST"});loadBots();}
let logTimer=null;
function botLogs(n){
  $("#modal").style.display="flex"; $("#modalTitle").textContent="📜 Logs · "+n;
  clearInterval(logTimer);
  const poll=async()=>{try{const d=await api(`/api/bots/${n}/logs?lines=250`);$("#modalPre").textContent=(d.lines||[]).join("\n")||"(log trống)";}catch(e){}};
  poll(); logTimer=setInterval(poll,1500);
}
$("#modalClose").onclick=()=>{$("#modal").style.display="none";clearInterval(logTimer);};
$("#botForm").onsubmit=async ev=>{
  ev.preventDefault();
  const fd=new FormData($("#botForm"));
  try{await api("/api/bots/upload",{method:"POST",body:fd});toast("✅ Đã lưu bot");$("#botForm").reset();loadBots();}
  catch(e){toast("❌ "+e.message);}
};

/* ── files ── */
let curPath="";
function fmtSize(s){if(s>1048576)return (s/1048576).toFixed(1)+" MB";if(s>1024)return (s/1024).toFixed(1)+" KB";return s+" B";}
async function loadFiles(p){
  if(p!==undefined) curPath=p;
  try{
    const d=await api("/api/files?path="+encodeURIComponent(curPath));
    curPath=d.path; $("#filePath").value=curPath;
    $("#fileRows").innerHTML=(d.items||[]).map(f=>`<tr>
      <td>${f.is_dir?"📁":"📄"} <a class="flink" href="javascript:void 0" onclick="${f.is_dir?`navTo('${f.name}')`:`downloadFile('${f.name}')`}">${f.name}</a></td>
      <td>${f.is_dir?"—":fmtSize(f.size)}</td>
      <td>${f.is_dir?`<button class="btn danger" onclick="delFile('${f.name}')">🗑️</button>`
        :`<button class="btn" onclick="downloadFile('${f.name}')">⬇️</button> <button class="btn danger" onclick="delFile('${f.name}')">🗑️</button>`}</td></tr>`).join("")
      || `<tr><td colspan="3" class="mut">(trống)</td></tr>`;
  }catch(e){toast("❌ "+e.message);}
}
function navTo(n){loadFiles(curPath?curPath+"/"+n:n);}
function downloadFile(n){
  const p=curPath?curPath+"/"+n:n;
  const u=new URL("/api/files/download?path="+encodeURIComponent(p),location.href);
  if(KEY)u.searchParams.set("key",KEY); window.open(u,"_blank");
}
async function delFile(n){if(!confirm("Xóa "+n+"?"))return;await api("/api/files/delete",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({path:curPath?curPath+"/"+n:n})});loadFiles(curPath);}
$("#fileGo").onclick=()=>loadFiles($("#filePath").value);
$("#btnUpFile").onclick=()=>$("#fileInput").click();
$("#fileInput").onchange=async()=>{
  const f=$("#fileInput").files[0]; if(!f)return;
  const fd=new FormData(); fd.append("file",f); fd.append("path",curPath);
  await api("/api/files/upload",{method:"POST",body:fd});
  toast("✅ Đã upload"); $("#fileInput").value=""; loadFiles(curPath);
};
$("#btnMkdir").onclick=async()=>{
  const n=prompt("Tên thư mục mới:"); if(!n)return;
  await api("/api/files/mkdir",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({path:(curPath?curPath+"/":"")+n})});
  loadFiles(curPath);
};

/* ── boot ── */
loadStats(); setInterval(loadStats,3000);
loadTunnels(); setInterval(loadTunnels,6000);
loadServices(); setInterval(loadServices,6000);
loadBots(); setInterval(loadBots,6000);
pollActions(); setInterval(pollActions,4000);
loadFiles();
</script>
</body></html>
'''

_index_cache = None
async def index(request):
    global _index_cache
    if _index_cache is None:
        _index_cache = DASH_HTML.replace("__TOKEN__", TOKEN)
    return web.Response(text=_index_cache, content_type="text/html")

async def api_health(request):
    return web.json_response({"ok": True, "env": detect_env(), "port": PORT})

def build_app():
    app = web.Application(middlewares=[auth_mw], client_max_size=1024 ** 3)
    app.router.add_get("/", index)
    app.router.add_get("/api/health", api_health)
    app.router.add_get("/api/stats", api_stats)
    app.router.add_get("/api/tunnels", api_tunnels)
    app.router.add_get("/api/services", api_services)
    app.router.add_get("/api/bots", api_bots)
    app.router.add_post("/api/bots/upload", api_bots_upload)
    app.router.add_post("/api/bots/{name}/start", api_bot_start)
    app.router.add_post("/api/bots/{name}/stop", api_bot_stop)
    app.router.add_get("/api/bots/{name}/logs", api_bot_logs)
    app.router.add_post("/api/bots/{name}/delete", api_bot_delete)
    app.router.add_get("/api/files", api_files)
    app.router.add_get("/api/files/download", api_files_download)
    app.router.add_post("/api/files/upload", api_files_upload)
    app.router.add_post("/api/files/delete", api_files_delete)
    app.router.add_post("/api/files/mkdir", api_files_mkdir)
    app.router.add_post("/api/action", api_action)
    app.router.add_get("/api/action/logs", api_action_logs)
    return app

if __name__ == "__main__":
    print(f"🎮 GPU Cloud Server dashboard · port {PORT} · auth={'ON' if AUTH_ENABLED else 'OFF'} · env={detect_env()}")
    web.run_app(build_app(), host="0.0.0.0", port=PORT, print=lambda *a: None, access_log=None)


In [ ]:
#@title 🎮 Bước 2 · Khởi động GPU Web Server + Cloudflare Tunnel { display-mode: "form" }
# ▶ GIỮ Ô NÀY CHẠY LIÊN TỤC — đây là nơi server sống. URL public sẽ in ra dưới đây.
# ✅ Chạy lại được nhiều lần (tự dọn tiến trình cũ). Nếu thiếu Desktop: chạy ô 🩺 Sửa nhanh rồi chạy lại ô này.
import os, sys, json, time, re, shutil, socket, subprocess, threading, secrets, pathlib, importlib.util

HOME = pathlib.Path.home()
GDIR, BDIR, LDIR = HOME / "gpu_server", HOME / "bots", HOME / "bot_logs"
for d in (GDIR, BDIR, LDIR):
    d.mkdir(parents=True, exist_ok=True)
LOCAL_BIN = str(HOME / ".local" / "bin")
pathlib.Path(LOCAL_BIN).mkdir(parents=True, exist_ok=True)
os.environ["PATH"] = LOCAL_BIN + os.pathsep + os.environ.get("PATH", "")

TOKEN = None
try:
    TOKEN = json.loads((GDIR / "server_info.json").read_text()).get("token")
except Exception:
    pass
TOKEN = TOKEN or secrets.token_hex(8)

PORT_DASH, PORT_NOVNC, PORT_TTYD, DISPLAY = 8080, 6080, 7681, ":1"
TUNNELS_FILE = pathlib.Path("/tmp/tunnels.json")
TUNNELS, PROCS = {}, []
TL = threading.Lock()

def have(c): return shutil.which(c) is not None

def port_open(port, timeout=0.5):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=timeout):
            return True
    except OSError:
        return False

def wait_port(port, timeout=25):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if port_open(port):
            return True
        time.sleep(0.4)
    return False

def kill_matching(needles):
    me = os.getpid()
    for d in pathlib.Path("/proc").iterdir():
        if not d.name.isdigit() or int(d.name) == me:
            continue
        try:
            cmd = (d / "cmdline").read_bytes().decode("utf-8", "ignore").replace("\x00", " ")
        except Exception:
            continue
        if any(n in cmd for n in needles):
            try:
                os.kill(int(d.name), 15)
            except Exception:
                pass

def save_tunnels():
    try:
        with TL:
            TUNNELS_FILE.write_text(json.dumps(TUNNELS))
    except Exception:
        pass

def spawn(args, name, gui=False, extra_env=None):
    env = dict(os.environ)
    if gui:
        env["DISPLAY"] = DISPLAY
    if extra_env:
        env.update(extra_env)
    logf = GDIR / f"{name}.log"
    lf = open(logf, "ab", buffering=0)
    lf.write(f"\n══════ ▶ {name} · {time.strftime('%H:%M:%S')} ══════\n".encode())
    p = subprocess.Popen([str(a) for a in args], stdout=lf, stderr=subprocess.STDOUT,
                         stdin=subprocess.DEVNULL, env=env, start_new_session=True)
    PROCS.append(p)
    print(f"  ▶ {name:<11} pid={p.pid:<6} log={logf}")
    return p

def start_tunnel(label, port, attempts=2):
    cf = shutil.which("cloudflared")
    if not cf:
        print(f"  ⚠️ [{label}] thiếu cloudflared → không tạo tunnel")
        return
    tail = deque(maxlen=10)
    for attempt in range(1, attempts + 1):
        found = threading.Event()
        p = subprocess.Popen([cf, "tunnel", "--url", f"http://localhost:{port}",
                              "--no-autoupdate", "--protocol", "http2"],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, start_new_session=True)
        PROCS.append(p)
        def reader(p=p, found=found, tail=tail):
            for line in p.stdout:
                tail.append(line.rstrip())
                m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
                if m:
                    with TL:
                        TUNNELS[label] = m.group(0)
                    save_tunnels()
                    found.set()
        threading.Thread(target=reader, daemon=True).start()
        if found.wait(25) and TUNNELS.get(label):
            print(f"  🌐 [{label}] {TUNNELS[label]}")
            return
        try:
            p.terminate()
        except Exception:
            pass
        if attempt < attempts:
            print(f"  ⚠️ [{label}] lần {attempt} chưa tạo được URL — thử lại…")
            time.sleep(2)
    print(f"  ❌ [{label}] không tạo được tunnel sau {attempts} lần. Log cuối:")
    for l in list(tail)[-6:]:
        print("      " + l[:150])

print("🧹 Dọn tiến trình cũ (nếu có)…")
kill_matching(["Xvfb", "x11vnc", "websockify", "ttyd", "cloudflared", "dashboard.py"])
time.sleep(1.5)
TUNNELS_FILE.write_text("{}")

# ── thành phần desktop ──
WS_CMD = ["websockify"] if have("websockify") else (
    [sys.executable, "-m", "websockify"] if importlib.util.find_spec("websockify") else None)
NOVNC_WEB = ("/usr/share/novnc" if os.path.isdir("/usr/share/novnc")
             else (str(HOME / "novnc") if (HOME / "novnc").is_dir() else None))
DESKTOP_OK = (have("Xvfb") and have("x11vnc") and bool(WS_CMD) and bool(NOVNC_WEB)
              and (have("startxfce4") or have("xterm")))

print("🧩 Thành phần:")
for label, ok in [("Xvfb (desktop ảo)", have("Xvfb")), ("x11vnc", have("x11vnc")),
                  ("websockify", bool(WS_CMD)), ("noVNC web", bool(NOVNC_WEB)),
                  ("XFCE/xterm", have("startxfce4") or have("xterm")),
                  ("ttyd", have("ttyd")), ("cloudflared", have("cloudflared"))]:
    print(f"  {'✅' if ok else '❌'} {label}")
if not DESKTOP_OK:
    print("  ⚠️ Thiếu thành phần Desktop → chạy ô 『🩺 SỬA NHANH』 rồi chạy lại ô này.")
    print("     (Vẫn tiếp tục với Dashboard / Bots / Terminal)")

print("🎬 Khởi động dịch vụ…")
if DESKTOP_OK:
    spawn(["Xvfb", DISPLAY, "-screen", "0", "1600x900x24", "-nolisten", "tcp"], "xvfb")
    for _ in range(20):
        if os.path.exists("/tmp/.X11-unix/X1"):
            break
        time.sleep(0.5)
    if have("xsetroot"):
        spawn(["xsetroot", "-solid", "#131022"], "xsetroot", gui=True)
    if have("dbus-run-session") and have("startxfce4"):
        spawn(["dbus-run-session", "--", "startxfce4"], "xfce4", gui=True)
    elif have("xterm"):
        spawn(["xterm", "-maximized"], "xterm", gui=True)
    spawn(["x11vnc", "-display", DISPLAY, "-rfbport", "5901", "-forever", "-shared",
           "-nopw", "-noxrecord", "-noxfixes", "-noxdamage"], "x11vnc")
    spawn(WS_CMD + ["--web", NOVNC_WEB, PORT_NOVNC, "localhost:5901"], "websockify")
else:
    print("  ⚠️ bỏ qua Desktop")

if have("ttyd"):
    spawn(["ttyd", "-W", "-p", PORT_TTYD, "bash", "-l"], "ttyd")
else:
    print("  ⚠️ thiếu ttyd → không có Web Terminal")

dash_py = next((c for c in [pathlib.Path.cwd() / "dashboard.py", HOME / "dashboard.py",
                            GDIR / "dashboard.py"] if c.exists()), None)
if dash_py:
    spawn([sys.executable, str(dash_py)], "dashboard",
          extra_env={"DASH_PORT": str(PORT_DASH), "DASH_TOKEN": TOKEN,
                     "TUNNELS_FILE": str(TUNNELS_FILE), "PYTHONUNBUFFERED": "1"})
else:
    print("  ⚠️ không tìm thấy dashboard.py — chạy lại ô %%writefile dashboard.py rồi chạy ô này")

if not wait_port(PORT_DASH, 25):
    print("  ⚠️ Dashboard chưa mở cổng — xem log:", GDIR / "dashboard.log")
start_tunnel("dashboard", PORT_DASH)

if DESKTOP_OK:
    if wait_port(PORT_NOVNC, 25):
        start_tunnel("desktop", PORT_NOVNC)
    else:
        print(f"  ❌ noVNC không mở cổng {PORT_NOVNC} — xem {GDIR}/websockify.log, x11vnc.log, xfce4.log")
if have("ttyd"):
    if wait_port(PORT_TTYD, 15):
        start_tunnel("terminal", PORT_TTYD)
    else:
        print(f"  ❌ ttyd không mở cổng {PORT_TTYD} — xem {GDIR}/ttyd.log")

demo = BDIR / "demo_bot.py"
if not demo.exists():
    demo.write_text("""import time
print("🤖 Demo bot đã khởi động!", flush=True)
while True:
    print(f"💓 demo bot alive · {time.strftime('%H:%M:%S')}", flush=True)
    time.sleep(60)
""")

for _ in range(45):
    if TUNNELS.get("dashboard") and (TUNNELS.get("desktop") or not DESKTOP_OK):
        break
    time.sleep(1)

dash = TUNNELS.get("dashboard")
desk = TUNNELS.get("desktop")
term = TUNNELS.get("terminal")
dash_url = f"{dash}/?key={TOKEN}" if dash else f"(thất bại — chạy lại ô này)"
desk_url = f"{desk}/vnc.html?autoconnect=1&resize=scale&reconnect=1" if desk else "(thất bại — xem log bên trên)"
term_url = term or "(thất bại — xem log bên trên)"

(GDIR / "server_info.json").write_text(json.dumps(
    {"token": TOKEN, "tunnels": dict(TUNNELS),
     "ports": {"dashboard": PORT_DASH, "desktop": PORT_NOVNC, "terminal": PORT_TTYD}}, indent=2))

print()
print("═" * 66)
print("   🎮  GPU CLOUD SERVER ĐANG CHẠY — GIỮ Ô NÀY CHẠY LIÊN TỤC!")
print("═" * 66)
print(f"   📊 Dashboard     {dash_url}")
print(f"   🖥️  Cloud Desktop  {desk_url}")
print(f"   ⌨️  Web Terminal   {term_url}")
print(f"   🔑 Access Key     {TOKEN}")
print("═" * 66)
print("   💡 URL MỚI khác URL cũ — cập nhật lại bookmark/tab đang mở!")
print("   💡 Kiểm tra trạng thái từng dịch vụ: Dashboard → tab Tổng quan → ⚙️ Dịch vụ nền")

n = 0
try:
    while True:
        time.sleep(30)
        n += 1
        if n % 40 == 0:
            print(f"💓 [{time.strftime('%H:%M:%S')}] server vẫn đang chạy — {dash_url}")
except KeyboardInterrupt:
    print("🛑 Đang dừng toàn bộ dịch vụ…")
    for p in PROCS:
        try:
            p.terminate()
        except Exception:
            pass


In [ ]:
#@title 🩺 SỬA NHANH · Chẩn đoán + cài thiếu + khởi động lại toàn bộ server { display-mode: "form" }
# ══ Dùng khi: Cloud Gaming không có tunnel / Desktop đen / thiếu dịch vụ ══
# CÁCH DÙNG: ① Interrupt (■) ô Bước 2 đang chạy  →  ② chạy ô này  →  ③ dùng URL MỚI in ra dưới đây.
import os, sys, json, time, re, shutil, socket, subprocess, threading, secrets, pathlib
import importlib.util, platform, urllib.request, zipfile
from collections import deque

HOME = pathlib.Path.home()
GDIR, BDIR, LDIR = HOME / "gpu_server", HOME / "bots", HOME / "bot_logs"
for d in (GDIR, BDIR, LDIR):
    d.mkdir(parents=True, exist_ok=True)
LOCAL_BIN = str(HOME / ".local" / "bin")
pathlib.Path(LOCAL_BIN).mkdir(parents=True, exist_ok=True)
os.environ["PATH"] = LOCAL_BIN + os.pathsep + os.environ.get("PATH", "")
root = os.geteuid() == 0
SUDO = "" if (root or not shutil.which("sudo")) else "sudo -n "

PORT_DASH, PORT_NOVNC, PORT_TTYD, DISPLAY = 8080, 6080, 7681, ":1"
TUNNELS_FILE = pathlib.Path("/tmp/tunnels.json")
TUNNELS, PROCS = {}, []
TL = threading.Lock()

def sh(cmd, timeout=None):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    return p.returncode, ((p.stdout or "") + (p.stderr or ""))

def have(c): return shutil.which(c) is not None

def port_open(port, timeout=0.5):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=timeout):
            return True
    except OSError:
        return False

def wait_port(port, timeout=25):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if port_open(port):
            return True
        time.sleep(0.4)
    return False

def kill_matching(needles):
    me = os.getpid()
    for d in pathlib.Path("/proc").iterdir():
        if not d.name.isdigit() or int(d.name) == me:
            continue
        try:
            cmd = (d / "cmdline").read_bytes().decode("utf-8", "ignore").replace("\x00", " ")
        except Exception:
            continue
        if any(n in cmd for n in needles):
            try:
                os.kill(int(d.name), 15)
            except Exception:
                pass

def comp(label, ok, extra=""):
    print(f"  {'✅' if ok else '❌'} {label}" + (f" — {extra}" if extra else ""))

def spawn(args, name, gui=False, extra_env=None):
    env = dict(os.environ)
    if gui:
        env["DISPLAY"] = DISPLAY
    if extra_env:
        env.update(extra_env)
    logf = GDIR / f"{name}.log"
    lf = open(logf, "ab", buffering=0)
    lf.write(f"\n══════ ▶ {name} · {time.strftime('%H:%M:%S')} ══════\n".encode())
    p = subprocess.Popen([str(a) for a in args], stdout=lf, stderr=subprocess.STDOUT,
                         stdin=subprocess.DEVNULL, env=env, start_new_session=True)
    PROCS.append(p)
    print(f"  ▶ {name:<11} pid={p.pid:<6} log={logf}")
    return p

def save_tunnels():
    try:
        with TL:
            TUNNELS_FILE.write_text(json.dumps(TUNNELS))
    except Exception:
        pass

def start_tunnel(label, port, attempts=2):
    cf = shutil.which("cloudflared")
    if not cf:
        print(f"  ⚠️ [{label}] thiếu cloudflared")
        return
    tail = deque(maxlen=10)
    for attempt in range(1, attempts + 1):
        found = threading.Event()
        p = subprocess.Popen([cf, "tunnel", "--url", f"http://localhost:{port}",
                              "--no-autoupdate", "--protocol", "http2"],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1, start_new_session=True)
        PROCS.append(p)
        def reader(p=p, found=found, tail=tail):
            for line in p.stdout:
                tail.append(line.rstrip())
                m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
                if m:
                    with TL:
                        TUNNELS[label] = m.group(0)
                    save_tunnels()
                    found.set()
        threading.Thread(target=reader, daemon=True).start()
        if found.wait(25) and TUNNELS.get(label):
            print(f"  🌐 [{label}] {TUNNELS[label]}")
            return
        try:
            p.terminate()
        except Exception:
            pass
        if attempt < attempts:
            print(f"  ⚠️ [{label}] lần {attempt} chưa tạo được URL — thử lại…")
            time.sleep(2)
    print(f"  ❌ [{label}] không tạo được tunnel sau {attempts} lần. Log cuối:")
    for l in list(tail)[-6:]:
        print("      " + l[:150])

TOKEN = None
try:
    TOKEN = json.loads((GDIR / "server_info.json").read_text()).get("token")
except Exception:
    pass
TOKEN = TOKEN or secrets.token_hex(8)

# ═══════════ 1. CHẨN ĐOÁN ═══════════
print("🔍 CHẨN ĐOÁN THÀNH PHẦN")
WS_BIN = have("websockify")
WS_MOD = importlib.util.find_spec("websockify") is not None
NOVNC_WEB = ("/usr/share/novnc" if os.path.isdir("/usr/share/novnc")
             else (str(HOME / "novnc") if (HOME / "novnc").is_dir() else None))
ok_xvfb, ok_vnc = have("Xvfb"), have("x11vnc")
ok_gui = have("startxfce4") or have("xterm")
comp("Xvfb (desktop ảo)", ok_xvfb)
comp("x11vnc (VNC server)", ok_vnc)
comp("websockify (VNC → web)", WS_BIN or WS_MOD,
     "" if WS_BIN else ("module pip (không có binary)" if WS_MOD else "thiếu hoàn toàn"))
comp("noVNC (web client)", NOVNC_WEB is not None, NOVNC_WEB or "")
comp("XFCE / xterm", ok_gui)
comp("ttyd (web terminal)", have("ttyd"))
comp("cloudflared (tunnel)", have("cloudflared"))

# ═══════════ 2. CÀI THIẾU ═══════════
need_apt = not (ok_xvfb and ok_vnc and ok_gui and NOVNC_WEB and (WS_BIN or WS_MOD))
if need_apt and have("apt-get"):
    pkgs = [n for n, ok in [("xvfb", ok_xvfb), ("x11vnc", ok_vnc), ("novnc", NOVNC_WEB is None),
                            ("websockify", not (WS_BIN or WS_MOD)),
                            ("dbus-x11", not have("dbus-run-session")),
                            ("x11-xserver-utils", not have("xsetroot")),
                            ("xterm", not have("xterm")), ("xfce4", not have("startxfce4")),
                            ("xfce4-terminal", not have("xfce4-terminal"))] if not ok]
    print(f"\n📦 apt cài: {' '.join(pkgs)} (có thể mất vài phút)")
    rc, out = sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get update -y", timeout=300)
    if rc != 0:
        print("  ⚠️ apt-get update lỗi (log cuối):\n" +
              "\n".join("    " + l for l in out.strip().splitlines()[-8:]))
    rc, out = sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends {' '.join(pkgs)}",
                 timeout=1500)
    if rc != 0:
        print("  ⚠️ Cài cả list bị lỗi — thử từng gói để tìm thủ phạm:")
        for pkg in pkgs:
            rc2, out2 = sh(f"{SUDO}DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends {pkg}",
                           timeout=600)
            print(f"    {'✅' if rc2 == 0 else '❌'} {pkg}")
            if rc2 != 0:
                print("\n".join("       " + l for l in out2.strip().splitlines()[-5:]))

if not have("websockify") and not WS_MOD:
    print("\n🐍 pip fallback: websockify")
    sh(f"{sys.executable} -m pip install -q websockify", timeout=300)
    WS_MOD = importlib.util.find_spec("websockify") is not None
    print(f"  {'✅' if WS_MOD else '❌'} websockify module pip")

if not os.path.isdir("/usr/share/novnc") and not (HOME / "novnc").is_dir():
    print("\n⬇️ Fallback: tải noVNC từ GitHub → ~/novnc")
    for zurl in ("https://github.com/novnc/noVNC/archive/refs/tags/v1.5.0.zip",
                 "https://github.com/novnc/noVNC/archive/refs/heads/master.zip"):
        try:
            ztmp = pathlib.Path("/tmp/novnc.zip")
            urllib.request.urlretrieve(zurl, ztmp)
            with zipfile.ZipFile(ztmp) as z:
                z.extractall("/tmp/_novnc_extract")
            src = next(pathlib.Path("/tmp/_novnc_extract").iterdir())
            dst = HOME / "novnc"
            if dst.exists():
                shutil.rmtree(dst)
            shutil.move(str(src), str(dst))
            NOVNC_WEB = str(dst)
            print(f"  ✅ {NOVNC_WEB}")
            break
        except Exception as e:
            print(f"  ⚠️ {zurl} → {e}")

if not have("ttyd"):
    arch = {"x86_64": "x86_64", "aarch64": "aarch64"}.get(platform.machine())
    if arch:
        dst = pathlib.Path(LOCAL_BIN) / "ttyd"
        try:
            urllib.request.urlretrieve(
                f"https://github.com/tsl0922/ttyd/releases/download/1.7.7/ttyd.{arch}", dst)
            os.chmod(dst, 0o755)
            print(f"\n✅ ttyd binary → {dst}")
        except Exception as e:
            print("\n❌ tải ttyd:", e)
if not have("cloudflared"):
    arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")
    dst = "/usr/local/bin/cloudflared" if root else pathlib.Path(LOCAL_BIN) / "cloudflared"
    try:
        urllib.request.urlretrieve(
            f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}", dst)
        os.chmod(dst, 0o755)
        print(f"\n✅ cloudflared → {dst}")
    except Exception as e:
        print("\n❌ tải cloudflared:", e)

# tổng kết sau cài đặt
WS_CMD = ["websockify"] if have("websockify") else (
    [sys.executable, "-m", "websockify"] if WS_MOD else None)
NOVNC_WEB = ("/usr/share/novnc" if os.path.isdir("/usr/share/novnc")
             else (str(HOME / "novnc") if (HOME / "novnc").is_dir() else None))
DESKTOP_OK = (have("Xvfb") and have("x11vnc") and bool(WS_CMD) and bool(NOVNC_WEB)
              and (have("startxfce4") or have("xterm")))
print("\n🧩 KẾT QUẢ:", "Desktop sẵn sàng ✅" if DESKTOP_OK
      else "Desktop vẫn thiếu ⚠️ — xem ❌ ở trên (Dashboard/Bots/Terminal vẫn chạy)")

# ═══════════ 3. DỌN + KHỞI ĐỘNG LẠI ═══════════
print("\n🧹 Dọn tiến trình cũ…")
kill_matching(["Xvfb", "x11vnc", "websockify", "ttyd", "cloudflared", "dashboard.py"])
time.sleep(2)
TUNNELS_FILE.write_text("{}")

print("🎬 Khởi động lại toàn bộ…")
if DESKTOP_OK:
    spawn(["Xvfb", DISPLAY, "-screen", "0", "1600x900x24", "-nolisten", "tcp"], "xvfb")
    for _ in range(20):
        if os.path.exists("/tmp/.X11-unix/X1"):
            break
        time.sleep(0.5)
    if have("xsetroot"):
        spawn(["xsetroot", "-solid", "#131022"], "xsetroot", gui=True)
    if have("dbus-run-session") and have("startxfce4"):
        spawn(["dbus-run-session", "--", "startxfce4"], "xfce4", gui=True)
    elif have("xterm"):
        spawn(["xterm", "-maximized"], "xterm", gui=True)
    spawn(["x11vnc", "-display", DISPLAY, "-rfbport", "5901", "-forever", "-shared",
           "-nopw", "-noxrecord", "-noxfixes", "-noxdamage"], "x11vnc")
    spawn(WS_CMD + ["--web", NOVNC_WEB, PORT_NOVNC, "localhost:5901"], "websockify")
else:
    print("  ⚠️ bỏ qua Desktop")
if have("ttyd"):
    spawn(["ttyd", "-W", "-p", PORT_TTYD, "bash", "-l"], "ttyd")
dash_py = next((c for c in [pathlib.Path.cwd() / "dashboard.py", HOME / "dashboard.py",
                            GDIR / "dashboard.py"] if c.exists()), None)
if dash_py:
    spawn([sys.executable, str(dash_py)], "dashboard",
          extra_env={"DASH_PORT": str(PORT_DASH), "DASH_TOKEN": TOKEN,
                     "TUNNELS_FILE": str(TUNNELS_FILE), "PYTHONUNBUFFERED": "1"})
else:
    print("  ⚠️ không tìm thấy dashboard.py — chạy lại ô %%writefile dashboard.py rồi chạy ô này")

if wait_port(PORT_DASH, 25):
    start_tunnel("dashboard", PORT_DASH)
else:
    print(f"  ❌ Dashboard không mở cổng {PORT_DASH} — xem {GDIR}/dashboard.log")
if DESKTOP_OK:
    if wait_port(PORT_NOVNC, 25):
        start_tunnel("desktop", PORT_NOVNC)
    else:
        print(f"  ❌ noVNC không mở cổng {PORT_NOVNC} — xem {GDIR}/websockify.log, x11vnc.log, xfce4.log")
if have("ttyd"):
    if wait_port(PORT_TTYD, 15):
        start_tunnel("terminal", PORT_TTYD)
    else:
        print(f"  ❌ ttyd không mở cổng {PORT_TTYD} — xem {GDIR}/ttyd.log")

demo = BDIR / "demo_bot.py"
if not demo.exists():
    demo.write_text("""import time
print("🤖 Demo bot đã khởi động!", flush=True)
while True:
    print(f"💓 demo bot alive · {time.strftime('%H:%M:%S')}", flush=True)
    time.sleep(60)
""")

for _ in range(45):
    if TUNNELS.get("dashboard") and (TUNNELS.get("desktop") or not DESKTOP_OK):
        break
    time.sleep(1)

dash = TUNNELS.get("dashboard")
desk = TUNNELS.get("desktop")
term = TUNNELS.get("terminal")
dash_url = f"{dash}/?key={TOKEN}" if dash else "(thất bại — chạy lại ô này)"
desk_url = f"{desk}/vnc.html?autoconnect=1&resize=scale&reconnect=1" if desk else "(thất bại — xem log bên trên)"
term_url = term or "(thất bại — xem log bên trên)"

(GDIR / "server_info.json").write_text(json.dumps(
    {"token": TOKEN, "tunnels": dict(TUNNELS),
     "ports": {"dashboard": PORT_DASH, "desktop": PORT_NOVNC, "terminal": PORT_TTYD}}, indent=2))

print()
print("═" * 66)
print("   🩺  SỬA XONG — GIỮ Ô NÀY CHẠY LIÊN TỤC (thay cho ô Bước 2)!")
print("═" * 66)
print(f"   📊 Dashboard     {dash_url}")
print(f"   🖥️  Cloud Desktop  {desk_url}")
print(f"   ⌨️  Web Terminal   {term_url}")
print(f"   🔑 Access Key     {TOKEN}")
print("═" * 66)
print("   ⚠️ URL MỚI khác URL cũ — mở link Dashboard mới này, đừng dùng link cũ!")
print("   💡 Xem trạng thái dịch vụ: Dashboard → Tổng quan → ⚙️ Dịch vụ nền")

n = 0
try:
    while True:
        time.sleep(30)
        n += 1
        if n % 40 == 0:
            print(f"💓 [{time.strftime('%H:%M:%S')}] server vẫn đang chạy — {dash_url}")
except KeyboardInterrupt:
    print("🛑 Đang dừng toàn bộ dịch vụ…")
    for p in PROCS:
        try:
            p.terminate()
        except Exception:
            pass


## 🎉 Server đang chạy — dùng thế nào?

### 📊 Tab Tổng quan
Giám sát GPU (tên card, % util, VRAM, nhiệt độ), CPU, RAM, disk, uptime + 3 URL public để copy.

### 🎮 Tab Cloud Gaming (tính năng hay nhất)
1. Bấm **🖥️ Mở Cloud Desktop** → mở desktop XFCE trong tab mới (noVNC)
2. Bấm **🎮 Cài Steam** trên dashboard (chạy ngầm, xem log bên dưới)
3. Vào desktop → mở Steam → đăng nhập → cài & chơi game
4. Muốn cài gì thêm: ô "Lệnh tuỳ ý" hoặc **⌨️ Web Terminal**

### 🤖 Tab Bots — treo bot 24/7 (trong lúc phiên sống)
- **Upload** file `.py` + ô Env chứa token (`DISCORD_TOKEN=xxx`, mỗi dòng một biến)
- **▶ Start / ■ Stop**, xem **📜 Logs** realtime, auto có thư viện sẵn: cài qua nút ⚡
- Bot mẫu `demo_bot` được tạo sẵn để bạn thử

Ví dụ bot Discord:

```python
import os, discord
from discord.ext import commands
intents = discord.Intents.default(); intents.message_content = True
bot = commands.Bot(command_prefix="!", intents=intents)

@bot.event
async def on_ready(): print("✅", bot.user)

@bot.command()
async def ping(ctx): await ctx.send("🏓 Pong!")

bot.run(os.environ["DISCORD_TOKEN"])
```

### 📁 Tab Files
Upload/download/xoá file trên VM — tiện khi cần đưa model/save game lên.

### ⌨️ Web Terminal
bash đầy đủ trên VM — cài bất cứ thứ gì (`pip`, `apt`, `git`, …).

---

### 🩺 Có sự cố? (Cloud Gaming không có tunnel / Desktop đen / thiếu dịch vụ)
1. **Interrupt (■)** ô Bước 2 đang chạy
2. Chạy ô **🩺 SỬA NHANH** — tự chẩn đoán từng thành phần, cài phần thiếu (kể cả fallback pip/GitHub khi apt hỏng), khởi động lại toàn bộ và in URL mới
3. Theo dõi trạng thái từng dịch vụ ở tab **Tổng quan → ⚙️ Dịch vụ nền** (bấm 📜 để xem log)

### 💡 Lưu ý quan trọng
- **Giữ ô Bước 2 chạy** — đóng tab/idle lâu Colab sẽ ngắt, server chết → chạy lại từ Bước 2 (URL mới).
- **URL tunnel là tạm thời** — mỗi phiên chạy có URL mới.
- **Đừng share URL desktop/terminal** (không có key bảo vệ như dashboard).
- **Giới hạn phiên:** Colab free ~3–12h, Kaggle ~9–12h/phiên & ~30h/tuần.
- **Disclaimer:** chạy game/bot nền có thể vi phạm ToS Colab/Kaggle — dùng có trách nhiệm, tự chịu rủi ro.

Repo: https://github.com/tiendatmeow-source/cloud-gaming-gpu · Landing: https://tiendatmeow-source.github.io/cloud-gaming-gpu/
